In [30]:
from datetime import datetime
from dateutil.relativedelta import relativedelta
import pandas as pd
import numpy as np
import pytz
from API.get_data.api_yahoo import get_historical_data, get_analyst_price_targets, get_business_recommendation, get_business_grades
from utils.trends_slope import check_trend_of_price
from utils.trends import get_trends_events
from utils.old_slope import qualify_slope
from collections import Counter


In [5]:
df = get_historical_data("MSFT", "5y")

In [6]:
df2=df.copy()

In [95]:
today = datetime.now()#.strftime('%Y-%m-%d')
first_of_this_month = today.replace(day=1)
first_of_last_month = first_of_this_month - relativedelta(months=1)

# Remettre l'heure à zéro (minuit)
first_of_this_month_midnight = first_of_this_month.replace(hour=0, minute=0, second=0, microsecond=0)
first_of_last_month_midnight = first_of_last_month.replace(hour=0, minute=0, second=0, microsecond=0)

one_year_rolling = today - relativedelta(months=12)
one_year_rolling_midnight = one_year_rolling.replace(hour=0, minute=0, second=0, microsecond=0)

five_year_rolling = today - relativedelta(year=5)
five_year_rolling_midnight = five_year_rolling.replace(hour=0, minute=0, second=0, microsecond=0)

year_to_date_midnight = datetime(first_of_this_month.year, 1, 1, 0, 0)

# Reset date to LocalTime 
df["Date"] = df["Date"].apply(lambda x: x.tz_localize(None))

# filter Dataframe
this_month_data = df[df["Date"]>=first_of_this_month_midnight]
last_month_data = df[df["Date"]>=first_of_last_month_midnight]
year_to_date_data = df[df["Date"]>=year_to_date_midnight]
one_year_rolling_period_data = df[df["Date"]>=one_year_rolling_midnight]
five_year_rolling_period_data = df[df["Date"]>=five_year_rolling_midnight]

In [8]:
different_dataframe_filter_bydate = [this_month_data, last_month_data, year_to_date_data, one_year_rolling_period_data, five_year_rolling_period_data]

In [9]:
def check_most_present_value_df(df_name):
    value_of_direction = df_name["EMA_Direction_Slope"].values
    most_present_direction = Counter(value_of_direction).most_common(1)[0][0]
    return most_present_direction

In [23]:
def extract_trend(df_name, col_name: str, degre: int) -> str:
    """
    Computes and qualifies the slope trend of a given column in the DataFrame.

    Args:
        df_name (pd.DataFrame): Filtered DataFrame to analyze.
        col (str): Column name to compute the trend on.
        degre (int): Degree for the polynomial fit. Default is 3.

    Returns:
        str: The trend label : Very bullish, Bullish, Stagnant, Bearish, Very bearish
    """
    slope = get_trends_events(df=df_name, cols=[col_name], degre=degre)
    slope_value = slope[f"{col_name}_slope"]
    return qualify_slope(slope_value)["steepness"]


In [ ]:
def today_analyse_price(df):
    return_df = df[["Date", "Close", "Volume"]].tail(1)
    different_dataframe_filter_bydate = [this_month_data, last_month_data, year_to_date_data, one_year_rolling_period_data, five_year_rolling_period_data]
    all_trends = []
    all_volume = []
    # compute trend price and volume
    for df_filter in different_dataframe_filter_bydate:
        check_trend_of_price(df_filter)
        trend_price = extract_trend(df_name=df_filter, col_name="Close", degre=3)
        all_trends.append(trend_price)
        trend_volume = extract_trend(df_name=df_filter, col_name="Volume", degre=3)
        all_volume.append(trend_volume)

    # allocate the most sxitch there have been during an period (up, down, flat)
    return_df["this_month_most_direction"] = check_most_present_value_df(this_month_data)
    return_df["last_month_most_direction"] = check_most_present_value_df(last_month_data)
    return_df["year_to_date_direction"] = check_most_present_value_df(year_to_date_data)
    return_df["one_year_rolling_period_most_direction"] = check_most_present_value_df(one_year_rolling_period_data)
    return_df["five_year_rolling_period_most_direction"] = check_most_present_value_df(five_year_rolling_period_data)

    # Allocate all trends calculate
    return_df["this_month_trend"] = all_trends[0]
    return_df["last_month_trend"] = all_trends[1]
    return_df["year_to_date_trend"] = all_trends[2]
    return_df["one_year_rolling_trend"] = all_trends[3]
    return_df["five_year_rollingtrend"] = all_trends[4]

    return_df["this_month_volume"] = all_volume[0]
    return_df["last_month_volume"] = all_volume[1]
    return_df["year_to_date_volume"] = all_volume[2]
    return_df["one_year_rolling_volume"] = all_volume[3]
    return_df["five_year_rolling_volume"] = all_volume[4]

    # Analyst recommendation
    df_business_recommendation = get_business_recommendation("MSFT")
    rating_of_month = df_business_recommendation[["strongBuy", "buy", "hold", "sell", "strongSell"]].loc[0]
    best_rating = rating_of_month.idxmax()
    return_df["analyst_recommendation_of_the_month"] = best_rating # ["strongBuy", "buy", "hold", "sell", "strongSell"]
    
    return return_df

In [29]:
analyse = today_analyse_price(df)
analyse.T

c:\Users\cleme\Documents\Ynov\M2\Projet Master\Projet Bourse\NEW\Projet_master\SRC\utils\trends_slope.py:49: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_name["EMA"] = df_name["Close"].ewm(span=ema_period, adjust=False).mean()
c:\Users\cleme\Documents\Ynov\M2\Projet Master\Projet Bourse\NEW\Projet_master\SRC\utils\trends_slope.py:52: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_name["EMA_Slope"] = (
c:\Users\cleme\Documents\Ynov\M2\Projet Master\Projet Bourse\NEW\Projet_master\SRC\utils\trends_slope

,1255
Date,2025-07-07 00:00:00
Close,497.51001
Volume,3330446
this_month_most_direction,Up
last_month_most_direction,Up
year_to_date_direction,Up
one_year_rolling_period_most_direction,Up
five_year_rolling_period_most_direction,Up
this_month_trend,Very bullish
last_month_trend,Very bullish


---

In [ ]:
# fonction a rajouter a get info company preporcessing -- info business

In [146]:
business_grades_tab = get_business_grades("MSFT")

In [101]:
# business_grades_tab[business_grades_tab["GradeDate"]>one_year_rolling_midnight]

In [156]:
all_firm = business_grades_tab.drop_duplicates(subset="Firm", keep='first')
all_firm[all_firm["GradeDate"]>one_year_rolling_midnight]

,index,GradeDate,Firm,ToGrade,FromGrade,Action,priceTargetAction,currentPriceTarget,GradeYear,GradeMonth,GradeDay
0,0,2025-06-26 13:58:40,Morgan Stanley,1,Overweight,main,Raises,530.0,2025,6,26
1,1,2025-06-25 14:50:42,Wedbush,1,Outperform,main,Raises,600.0,2025,6,25
2,2,2025-06-25 13:42:46,Wells Fargo,1,Overweight,main,Raises,585.0,2025,6,25
4,4,2025-06-11 14:54:00,Citigroup,1,Buy,main,Raises,605.0,2025,6,11
5,5,2025-05-23 15:03:26,RBC Capital,1,Outperform,reit,Maintains,525.0,2025,5,23
6,6,2025-05-22 15:02:04,Evercore ISI Group,1,Outperform,main,Raises,515.0,2025,5,22
7,7,2025-05-21 16:47:59,Goldman Sachs,1,Buy,main,Raises,550.0,2025,5,21
9,9,2025-05-01 21:57:32,Jefferies,1,Buy,main,Raises,550.0,2025,5,1
10,10,2025-05-01 21:42:13,DA Davidson,1,Buy,main,Raises,500.0,2025,5,1
13,13,2025-05-01 16:27:21,Stifel,1,Buy,main,Raises,500.0,2025,5,1


In [ ]:
def breakdown_of_analyst_recommendation(business_ticker):
    business_grades_tab = get_business_grades(business_ticker)
    last_recommendation_by_firm = business_grades_tab.drop_duplicates(subset="Firm", keep='first')
    lst_recom_one_year_rolling = last_recommendation_by_firm[last_recommendation_by_firm["GradeDate"]>one_year_rolling_midnight]
        

In [159]:
# Remplacement des mentions textuelles spécifiques par une note équivalente (ici 1)
replace_map = {
    "Sector Weight": 1,
    "Sector Outperform": 1
}

# Application des remplacements uniquement sur les valeurs correspondantes
all_firm.loc[:, "ToGrade"] = all_firm["ToGrade"].replace(replace_map)
all_firm.loc[:, "ToGrade"] = pd.to_numeric(all_firm["ToGrade"], errors="coerce").fillna(pd.NA).astype(int)

In [160]:
# Moyenne pondérée des ToGrade pour refléter le consensus global
sentiment_moyen = all_firm["ToGrade"].mean()
print(f"Sentiment moyen des analystes : {sentiment_moyen:.2f}")

# Pourcentage d’avis positifs, neutres et négatifs
positifs = (all_firm["ToGrade"] == 1).mean() * 100
negatifs = (all_firm["ToGrade"] == 0).mean() * 100

print(f"Répartition des recommandations :\n"
      f"    Positives : {positifs:.1f}%\n"
      f"    negatifs : {negatifs:.1f}%\n",
      f"    Total : {positifs+negatifs}")

Sentiment moyen des analystes : 0.81
Répartition des recommandations :
    Positives : 80.6%
    negatifs : 19.4%
     Total : 100.0


In [175]:
a ={
    "breakdown_of_analyst":
    {
        "distribution_of_recommendations" : {
            "positif": format(positifs, '.2f'),
            "negatifs": format(negatifs, '.2f')
        },
        "price_targets" : get_analyst_price_targets("AAPL")
    }
}

In [181]:
a

{'breakdown_of_analyst': {'distribution_of_recommendations': {'positif': '80.60',
   'negatifs': '19.40'},
  'price_targets': {'current': 209.95,
   'high': 300.0,
   'low': 170.62,
   'mean': 228.60326,
   'median': 230.0}}}

In [34]:
get_analyst_price_targets("AAPL")

{'current': 212.29,
 'high': 300.0,
 'low': 170.62,
 'mean': 228.60326,
 'median': 230.0}